<a href="https://colab.research.google.com/github/Denaro13/New-Hybrid-Block-Collocation-Method-Elliptic-PDEs/blob/main/New_Hybrid_Block_Collocation_Method.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Block Collocation Method
NHBCM (New Hybrid Block Collocation Method) Implementation
For solving elliptic PDEs:

$$b1*u_xx + b2*u_yy + b3*u_x + b4*u_y + b5*u = g(x,y)$$

Test Problem 1: Poisson Equation

Test Problem 2: Helmholtz Equation

In [1]:
import numpy as np
from scipy.optimize import fsolve
from scipy.sparse import csr_matrix, lil_matrix
from scipy.sparse.linalg import spsolve
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

### Test Problem 1: POISSON EQUATION

PDE: $\frac{\partial^{2}u}{\partial x ^{2}}+\frac{\partial^{2}u}{\partial y ^{2}} = xe^{y}$

Domain: $[0, 2]\times [0, 1]$

Exact Solution: $u(x,y)=xe^{y}$

In [4]:
class PoissonProblem:

    def __int__(self, N_x=10, N_y=5):
        """
        N_x: number of divisions in x-direction
        N_y: number of divisions in y-direction
        """

        self.N_x = N_x
        self.N_y = N_y
        self.x_min, self.x_max = 0, 2
        self.y_min, self.y_max = 0, 1

        # Create grids
        self.x = np.linspace(self.x_min, self.x_max, N_x + 1)
        self.y = np.linspace(self.y_min, self.y_max, N_y + 1)
        self.dx = self.x[1] - self.x[0]
        self.dy = self.y[1] - self.y[0]

        print(f"Problem 1 setup")
        print(f" Domain: [{self.x_min}, {self.x_max}] x [{self.y_min}, {self.y_max}]")
        print(f" Grid: {N_x+1} x {N_y+1} points")
        print(f" Step sizes: dx = {self.dx:.4f}, dy: {self.dy:.4f}")

        def exact_solution(self, x, y):
            """Exact solution: u(x,y) = x*exp(y)"""
            return x * np.exp(y)

        def boundary_conditions(self):
            """Apply Dirichlet boundary condtions"""
            u_boundary = np.zeros((len(self.x), len(self.y)))

            # Bottom Boundar: u(x, 0) =0
            u_boundary[:, 0] = 0

            # Top boundary: u(x, 1) = x*exp(1)
            u_boundary[:, -1] = self.x * np.exp(1)

            # Left boundary: u(0, y) = 0
            u_boundary[0, :] = 0

            # Right boundary: u(2, y) = 2*exp(y)
            u_boundary[-1, :] = 2 * np.exp(self.y)

            return u_boundary

        def ode_rhs(self, x_idx, y, u_at_y):
            """
            Right-hand side of ODE after MOL transformation

            After MOL, we habe d²u_i/dy² = f(y, u_i, others)

            where f = x_i*exp(y) - [u_{i+1}] - 2*u_i + u_{i-1}]/dx²
            """

            u = u_at_y # u[i] is solution at x_i

            # Finite difference approximation of d²u/dx²

            if x_idx == 0 or x_idx == len(self.x) -1:
                # Boundary points
                d2u_dx2 = 0
            else:
                d2u_dx2 = (u[x_idx + 1] - 2*u[x_idx] + u[x_idx - 1]) / (self.dx**2)

            # Right-hand side
            source = self.x[x_idx] + np.exp(y)

            # ODE: u'' = source - d²u/dx²
            f = source - d2u_dx2

            return f